# Import

In [ ]:
import numpy as np
import healpy as hp
import pandas as pd
import networkx as nx
import geopandas as gpd

from itertools import combinations

from pyproj import Geod
from shapely import union_all, intersects
from shapely.wkt import loads
from shapely.geometry import LineString
from healpix_geo.nested import healpix_to_lonlat
from healpix_geo.nested import lonlat_to_healpix

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

In [ ]:
def get_points(NSIDE):
    NPIX = hp.nside2npix(NSIDE)
    points = np.array(
        hp.pix2vec(NSIDE, range(NPIX), nest=True)
    ).T

    df = pd.DataFrame(points, columns=('x', 'y', 'z'))

    return df

def get_patches(NSIDE):
    NPIX = hp.nside2npix(NSIDE)

    pixels = np.arange(NPIX)
    patches = np.array(
        [hp.boundaries(NSIDE, i, nest=True).T for i in pixels]
    ).reshape(-1, 3)

    df = pd.DataFrame(patches, columns=('x', 'y', 'z'))

    return df

In [ ]:
def great_circle_path(a=None, b=None, n_points=100, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    points = geod.npts(
        a.x, a.y,
        b.x, b.y,
        n_points,
    )
    line = LineString(points)

    return line

def great_circle_distance(a=None, b=None, iterable=False):
    geod = Geod(ellps='WGS84')

    if iterable:
        (a, b) = a

    distance = geod.inv(
        a.x, a.y,
        b.x, b.y,
    )[2]

    return distance

# Data

## Borders

In [ ]:
kaz = gpd.read_file('data/geoBoundaries-KAZ-ADM0_simplified.geojson')
ukr = gpd.read_file('data/geoBoundaries-UKR-ADM0_simplified.geojson')
rus = gpd.read_file('data/geoBoundaries-RUS-ADM0_simplified.geojson')
blr = gpd.read_file('data/geoBoundaries-BLR-ADM0_simplified.geojson')
borders = pd.concat((kaz, ukr, rus, blr)).reset_index(drop=True)
borders = borders.to_crs(epsg=4326)
borders

## Airports

In [ ]:
airports = pd.read_csv('data/airports.csv')
airports = gpd.GeoDataFrame(
    airports.drop(['GeoPointLat', 'GeoPointLong'], axis=1),
    geometry=gpd.points_from_xy(
        *airports[['GeoPointLong', 'GeoPointLat']].values.T
    ), crs='EPSG:4326'
)
airports[airports.ICAO.isin(['KJFK', 'UAAA'])]

# HEALPix

In [ ]:
NSIDE = 16
healpix = get_points(NSIDE)

NPIX = hp.nside2npix(NSIDE)
ipix = np.arange(NPIX)
depth = np.log2(NSIDE).astype(int)

print(NPIX, depth)

In [ ]:
df = pd.read_csv('point_pairs.csv', index_col=False)

df = df.rename({'0': 'point_a', '1': 'point_b'}, axis=1)
df = df.drop(['Unnamed: 0'], axis=1)

df.loc[:, 'point_a'] = loads(df.loc[:, 'point_a'])
df.loc[:, 'point_b'] = loads(df.loc[:, 'point_b'])

df['healpix_id_a'] = lonlat_to_healpix(
    gpd.GeoSeries(df.loc[:, 'point_a']).geometry.x,
    gpd.GeoSeries(df.loc[:, 'point_a']).geometry.y,
    depth,
    ellipsoid='WGS84',
)
df['healpix_id_b'] = lonlat_to_healpix(
    gpd.GeoSeries(df.loc[:, 'point_b']).geometry.x,
    gpd.GeoSeries(df.loc[:, 'point_b']).geometry.y,
    depth,
    ellipsoid='WGS84',
)

df

# Graph

In [ ]:
shape_restricted = union_all(
    borders.loc[1:3, 'geometry']
)

start = lonlat_to_healpix(
    airports[airports.ICAO=='UAAA'].geometry.x,
    airports[airports.ICAO=='UAAA'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]
end = lonlat_to_healpix(
    airports[airports.ICAO=='KJFK'].geometry.x,
    airports[airports.ICAO=='KJFK'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]

In [ ]:
df['distance'] = df[['point_a', 'point_b']].apply(
    great_circle_distance,
    axis=1,
    raw=True,
    iterable=True,
)

In [ ]:
df.to_csv('point_pairs_full.csv', index=False)